In [1]:
import pandas as pd
import pandas as pd
import json
import re
import PyPDF2

In [2]:
def extractBooksAndPages(text):
    book_and_pages = {}
    start = re.search(r"Genesis\s*\.\.\.", text)
    if start:
        q = text[start.start():]  
    else:
        q = text

    pattern = r"([1-3]?\s?\w+(?: \w+)?)\s*\.\.\.\s*(\d+)"
    matches = re.findall(pattern, q)

    for book, page in matches:
        book_and_pages[book.strip()] = int(page)

    return book_and_pages

In [52]:
def getBookTitle(text, isStart):
    if isStart:
        #pattern = r"^Page\s+\d+\s+(?:\d+\s+)?([A-Za-z0-9]+(?:\s+[A-Za-z0-9]+)*)"  
        pattern = r"^Page\s+\d+\s+((?:\d+\s+)?[A-Za-z]+(?:\s+[A-Za-z]+)?)"
    else:
        pattern = r"^((?:\d+\s+)?[A-Za-z]+(?:\s+[A-Za-z]+)?)\s+[Pp]age\s+\d+"


    match = re.match(pattern, text)
    book_name = match.group(1).strip()
    book_name = re.sub(r"\s+The\b.*", "", book_name)
    return book_name

In [12]:
def removeHeaderText_First(text):
    pattern = r"\{1:1\}.*"
    match_text = re.search(pattern, text, re.DOTALL)
    return match_text.group()

In [13]:
def removeHeaderText_Second(text):
    if text.strip().startswith("Page"):
        cleaned_text = re.sub(r"^Page \d+ .*?\n", "", text.lstrip())
    else:
        cleaned_text = re.sub(r'^[A-Za-z0-9 ]+ Page \d+\n', "", text.lstrip())

    return cleaned_text

In [3]:
reader = PyPDF2.PdfReader('../pdf/The-Holy-Bible-King-James-Version.pdf')

In [4]:
contents_table = reader.pages[2] # 2 is where the table of contents is located
contents = extractBooksAndPages(contents_table.extract_text())
with open("../data/books.json", "w") as f:  
    json.dump(contents, f, indent=4)

In [ ]:
startPage = 21
start_of_chapter = False

for i in range(startPage, startPage+1):
    text = reader.pages[i].extract_text()

    if "{1:1}" in text:
        start_of_chapter = True

    if start_of_chapter:
        text = removeHeaderText_First(text)
    else:
        text = removeHeaderText_Second(text)

    print(text)

In [73]:
startPage = 723
start_of_chapter = False

for i in range(startPage, startPage+1):
    text = reader.pages[i].extract_text()

    if text.startswith("Page"):
        start_of_chapter = True
    else:
        start_of_chapter = False
    
    book = getBookTitle2(text, start_of_chapter)
    
    print(start_of_chapter)
    print(book)
    #print(text)

True
2 Peter The Second Epistle General


In [ ]:
import re

def getBookTitle2(text, isStart):
    text = text.strip()

    if isStart:
        # "Page <num> [optional number] <Word> [optional second word]" → Stop there
        pattern = r"^Page\s+\d+\s+((?:\d+\s+)?[A-Za-z]+(?:\s+[A-Za-z]+)?)"
    else:
        # "[optional number] <Word> [optional second word] Page <num>"
        pattern = r"^((?:\d+\s+)?[A-Za-z]+(?:\s+[A-Za-z]+)?)\s+[Pp]age\s+\d+"

    match = re.match(pattern, text)
    if match:
        return match.group(1).strip()
    return None
